# Human myoblast differentiation: GSE52529

This notebook adds the human skeletal muscle myoblast time course used in the original Monocle pseudotime study. It plots the observed 0, 24, 48, and 72 hour collection times and estimates diffusion pseudotime (DPT) from an early collection-time root. DPT is inferred from expression neighborhoods; it is not the observed collection time. GEO sample flags are used to exclude debris, control wells, and wells with anything other than one cell.

Source: Trapnell et al., [The dynamics and regulators of cell fate decisions are revealed by pseudotemporal ordering of single cells](https://doi.org/10.1038/nbt.2859); [GEO GSE52529](https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE52529).

In [ ]:
from pathlib import Path
import sys
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc

NOTEBOOK_DIR = Path.cwd() if (Path.cwd() / "src").is_dir() else Path.cwd() / "notebooks"
sys.path.insert(0, str(NOTEBOOK_DIR / "src"))
from biological_comparison import load_gse52529_myoblasts, make_imste, make_umap

RANDOM_STATE = 42
N_HVG = 1000
N_PCS = 50

## Load the processed single-cell matrix

The GEO matrix contains FPKM values (not raw counts). The loader keeps 306 clean single-cell profiles from 372 matrix columns, then applies `log1p`, highly variable gene selection, scaling, and PCA.


In [ ]:
myoblast = load_gse52529_myoblasts(random_state=RANDOM_STATE, n_hvg=N_HVG, n_pcs=N_PCS)
time_summary = (pd.DataFrame({"time_hours": myoblast["time_hours"]})
                .value_counts("time_hours").sort_index().rename("cells").reset_index())
display(time_summary)
X = myoblast["X"]
time_hours = myoblast["time_hours"]


## Compare IMSTE and UMAP by collection time

Both methods receive the same PCA coordinates. The observed time labels are only used to color the resulting plots.


In [ ]:
imste = make_imste(n_components=2, n_msts=4, n_epochs=1000, random_state=RANDOM_STATE)
umap = make_umap(n_components=2, n_neighbors=20, min_dist=0.1, random_state=RANDOM_STATE)
Z_imste = imste.fit_transform(X)
Z_umap = umap.fit_transform(X)

time_order = sorted(np.unique(time_hours))
colors = plt.get_cmap("viridis")(np.linspace(0.05, 0.95, len(time_order)))
color_by_time = {hour: colors[i] for i, hour in enumerate(time_order)}
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5), constrained_layout=True)
for ax, Z, title in zip(axes, (Z_imste, Z_umap), ("IMSTE", "UMAP")):
    for hour in time_order:
        mask = time_hours == hour
        ax.scatter(Z[mask, 0], Z[mask, 1], s=12, alpha=0.75, linewidths=0,
                   color=color_by_time[hour], label=f"{hour} h")
    ax.set_title(title)
    ax.set_xticks([]); ax.set_yticks([])
    ax.set_aspect("equal", adjustable="datalim")
axes[1].legend(title="Serum-switch time", frameon=False)
plt.show()

## Estimate diffusion pseudotime

Use the cell nearest the 0 hour PCA centroid as the root, then calculate DPT on a nearest-neighbor graph of the same PCA features. Collection time provides an external reference for interpreting the inferred ordering. A single DPT axis can simplify a branching process, so inspect the scatter and per-timepoint distributions rather than treating it as ground truth.


In [ ]:
trajectory = sc.AnnData(X=X.copy())
trajectory.obs["time_hours"] = time_hours
early = np.flatnonzero(time_hours == min(time_order))
early_center = X[early].mean(axis=0)
root_index = early[np.argmin(np.sum((X[early] - early_center) ** 2, axis=1))]
trajectory.uns["iroot"] = int(root_index)
sc.pp.neighbors(trajectory, n_neighbors=15, use_rep="X", random_state=RANDOM_STATE)
sc.tl.diffmap(trajectory)
sc.tl.dpt(trajectory, n_dcs=10)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5), constrained_layout=True)
for ax, Z, title in zip(axes, (Z_imste, Z_umap), ("IMSTE", "UMAP")):
    points = ax.scatter(Z[:, 0], Z[:, 1], c=trajectory.obs["dpt_pseudotime"],
                        cmap="plasma", s=12, alpha=0.8, linewidths=0)
    ax.set_title(f"{title}, colored by DPT")
    ax.set_xticks([]); ax.set_yticks([])
    ax.set_aspect("equal", adjustable="datalim")
fig.colorbar(points, ax=axes, label="Diffusion pseudotime", shrink=0.8)
plt.show()

dpt_by_time = (pd.DataFrame({"time_hours": time_hours,
                             "dpt": trajectory.obs["dpt_pseudotime"].to_numpy()})
              .groupby("time_hours")["dpt"]
              .agg(["count", "median", "min", "max"]))
display(dpt_by_time)

## Data notes

This is a processed FPKM matrix rather than raw UMI counts. Cell profiles were collected at four timepoints after serum switch. The original study used Monocle to recover a differentiation progression and also identified a non-differentiating contaminant branch. The DPT result here is a fresh analysis on this project’s PCA neighborhood graph, not a reproduction of the original Monocle trajectory. See [`data/gse52529/README.md`](../data/gse52529/README.md) for provenance and filtering details.